# EX: Probability Inference Engine

In this exercise, you will implement an automated probability reasoning engine in Python using NumPy and Pandas. The engine will ingest a joint probability distribution for a tactical scenario, calculate marginal probabilities, compute updated conditional probabilities given new intelligence observations, and verify statistical independence.

### Lab Steps:
An Airborne Warning and Control System (AWACS) classifies tracks based on two random variables:
1. **Identity ($I$):** Hostile ($H$) or Benign ($B$).
2. **IFF Transponder Response ($T$):** Valid Response ($V$) or Squawk Error ($E$).

In [1]:
import numpy as np
import pandas as pd

# Step 1: Define the Full Joint Probability Distribution Table
# Rows: Identity (Hostile, Benign)
# Columns: IFF Response (Valid, Error)
identities = ['Hostile', 'Benign']
iff_responses = ['Valid', 'Error']

# Probability values:
# Hostile & Valid: 0.01 (Hostile using captured IFF codes)
# Hostile & Error: 0.09 (Hostile with no/incorrect code)
# Benign & Valid:  0.85 (Friendly aircraft operating normally)
# Benign & Error:  0.05 (Friendly aircraft with transponder malfunction)
joint_prob_matrix = np.array([
    [0.01, 0.09],  # Hostile
    [0.85, 0.05]   # Benign
])

joint_df = pd.DataFrame(joint_prob_matrix, index=identities, columns=iff_responses)
print("=== Joint Probability Distribution Table P(Identity, IFF) ===")
print(joint_df)
print(f"Sum of all joint probabilities: {joint_df.values.sum():.2f}\n")

# Step 2: Marginalization (Summing Out Variables)
# P(Identity): Sum across columns (axis 1)
p_identity = joint_df.sum(axis=1)
print("=== Marginal Probabilities P(Identity) ===")
print(p_identity)

# P(IFF): Sum across rows (axis 0)
p_iff = joint_df.sum(axis=0)
print("\n=== Marginal Probabilities P(IFF) ===")
print(p_iff)
print()

# Step 3: Compute Conditional Probability P(Hostile | Error)
# P(Hostile | Error) = P(Hostile, Error) / P(Error)
joint_hostile_error = joint_df.loc['Hostile', 'Error']
marginal_error = p_iff['Error']
p_hostile_given_error = joint_hostile_error / marginal_error

print("=== Conditional Probability Inference ===")
print(f"P(Hostile, Error): {joint_hostile_error:.4f}")
print(f"P(Error):          {marginal_error:.4f}")
print(f"P(Hostile | Error) = {joint_hostile_error:.4f} / {marginal_error:.4f} = {p_hostile_given_error:.4f} ({p_hostile_given_error * 100:.2f}%)")
print()

# Step 4: Evaluate Statistical Independence
# Test if P(Hostile, Error) == P(Hostile) * P(Error)
marginal_hostile = p_identity['Hostile']
expected_independent_joint = marginal_hostile * marginal_error

print("=== Statistical Independence Evaluation ===")
print(f"Actual Joint P(Hostile, Error):           {joint_hostile_error:.4f}")
print(f"Expected Joint if Independent P(H)*P(E): {expected_independent_joint:.4f}")

is_independent = np.isclose(joint_hostile_error, expected_independent_joint)
print(f"Are Identity and IFF Response Independent? {'YES' if is_independent else 'NO (Dependent)'}")

=== Joint Probability Distribution Table P(Identity, IFF) ===
         Valid  Error
Hostile   0.01   0.09
Benign    0.85   0.05
Sum of all joint probabilities: 1.00

=== Marginal Probabilities P(Identity) ===
Hostile    0.1
Benign     0.9
dtype: float64

=== Marginal Probabilities P(IFF) ===
Valid    0.86
Error    0.14
dtype: float64

=== Conditional Probability Inference ===
P(Hostile, Error): 0.0900
P(Error):          0.1400
P(Hostile | Error) = 0.0900 / 0.1400 = 0.6429 (64.29%)

=== Statistical Independence Evaluation ===
Actual Joint P(Hostile, Error):           0.0900
Expected Joint if Independent P(H)*P(E): 0.0140
Are Identity and IFF Response Independent? NO (Dependent)



## Interpreting the Results

* **Marginal Base Rates:** The marginal distribution demonstrates that the airspace is predominantly benign: $P(\text{Hostile}) = 0.10$ and $P(\text{Benign}) = 0.90$. Across all flights, $14\%$ trigger an IFF squawk error ($P(\text{Error}) = 0.14$) due to either hostile spoofing or benign hardware malfunctions.

* **Belief Updating via Evidence:** When an IFF error is observed, conditioning on that evidence increases the probability of a hostile aircraft from a prior baseline of $10\%$ to a posterior probability of $64.29\%$ ($P(\text{Hostile} \mid \text{Error}) = 0.6429$). The remaining $35.71\%$ accounts for false alarms caused by friendly equipment failures.

* **Statistical Dependence as Sensor Utility:** The evaluation confirms that $P(\text{Hostile}, \text{Error}) = 0.0900 \neq P(\text{Hostile}) \times P(\text{Error}) = 0.0140$. Because the actual joint probability is significantly larger than the product of the marginals, identity and IFF squawk errors are statistically dependent, confirming the transponder provides strong diagnostic evidence rather than random noise.
